### Checks

`Urban Ward_Panel` sheets.


1. All TV_CODES should show up at least once. Either one shape for a town or village (NaN Ward code) or multiple ward shapes with codes.
2. Which TV_Codes have we been given multiple shapes for (wards)? Should match or overdeliver on their promise.

Add column that says "Delivery State": 
- BETTER - Ward boundary given but only TV or Subdistrict was expected
- GOOD - Ward boundary given as expected
- GOOD - Town/Village boundary given as expected
- GOOD - Subdistrict boundary given as expected
- BAD - Town/Village boundary given but Ward boundaries expected
- BAD - Subdistrict boundary given but Ward boundaries expected
- BAD - No boundary(s) given

## Setup

In [1]:
# set libraries to refresh
%load_ext autoreload
%autoreload 2

In [2]:
# general
from pathlib import Path
import pandas as pd
import geopandas as gpd

gpd.options.io_engine = "pyogrio"

In [3]:
from gridsample.utils import save_shapefiles

In [4]:
ROOT_DIR = Path("../")
DATA_DIR = ROOT_DIR / "data_panel"
RAW_DATA_DIR = DATA_DIR / "01. Raw Data"
CLEANED_DATA_DIR = (
    DATA_DIR
    / "02. Intermediate Outputs"
    / "replacements_v1"
    / "00. Merge and Quality Checks"
)
CLEANED_DATA_DIR.mkdir(parents=True, exist_ok=True)

## 0. Load request excel

In [5]:
excel_file = (
    RAW_DATA_DIR
    / "00. Boundary Requests"
    / "2025-09-23_SamplingOutput_Summary_Panel_Replacement_Wards.xlsx"
)

In [6]:
panel_df = pd.read_excel(excel_file, sheet_name="Urban Ward Level")

In [7]:
panel_df

,State,State_Name,District,District_Name,Subdistrict,Subd_Name,TownVillage,UrbanWardVillage,WardVillage_Name,TRU,WardVillage_Pop,Subd_Pop,State_Pop,WardVillageID,PCA_ID
0,2,HIMACHAL PRADESH,33,Shimla,182,Shimla (urban),800137,7,Shimla (M Corp.) WARD NO.-0007,Urban,8205,169578,6864602.0,033-00182-800137-0007,800137-0007
1,2,HIMACHAL PRADESH,33,Shimla,182,Shimla (urban),800137,17,Shimla (M Corp.) WARD NO.-0017,Urban,6526,169578,6864602.0,033-00182-800137-0017,800137-0017
2,2,HIMACHAL PRADESH,33,Shimla,182,Shimla (urban),800137,25,Shimla (M Corp.) WARD NO.-0025,Urban,6036,169578,6864602.0,033-00182-800137-0025,800137-0025
3,3,PUNJAB,37,Jalandhar,212,Jalandhar - I,800166,5,Jalandhar (M Corp.) WARD NO.-0005,Urban,11500,1145692,27743338.0,037-00212-800166-0005,800166-0005
4,3,PUNJAB,37,Jalandhar,212,Jalandhar - I,800166,10,Jalandhar (M Corp.) WARD NO.-0010,Urban,21042,1145692,27743338.0,037-00212-800166-0010,800166-0010
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
683,90,TELANGANA,537,Rangareddy,4524,Uppal,802918,5,GHMC (M Corp.) (Part) WARD NO.-0005,Urban,57688,384835,NaN,537-04524-802918-0005,802918-0005
684,90,TELANGANA,537,Rangareddy,4524,Uppal,802918,8,GHMC (M Corp.) (Part) WARD NO.-0008,Urban,75227,384835,NaN,537-04524-802918-0008,802918-0008
685,90,TELANGANA,538,Mahbubnagar,4568,Mahbubnagar,802922,6,Mahbubnagar (M) WARD NO.-0006,Urban,10515,249091,NaN,538-04568-802922-0006,802922-0006
686,90,TELANGANA,541,Khammam,4757,Khammam (Urban),579675,0,Erlapudi,Rural,12023,313504,NaN,541-04757-579675-0000,579675-0000


In [8]:
panel_df["Ward Boundary Available with MapSolve"] = "Yes"

In [9]:
# sort and reset index
panel_df = panel_df.sort_values(
    by=["State", "District", "Subdistrict", "TownVillage", "UrbanWardVillage"]
).reset_index(drop=True)

In [10]:
panel_df

,State,State_Name,District,District_Name,Subdistrict,Subd_Name,TownVillage,UrbanWardVillage,WardVillage_Name,TRU,WardVillage_Pop,Subd_Pop,State_Pop,WardVillageID,PCA_ID,Ward Boundary Available with MapSolve
0,2,HIMACHAL PRADESH,33,Shimla,182,Shimla (urban),800137,7,Shimla (M Corp.) WARD NO.-0007,Urban,8205,169578,6864602.0,033-00182-800137-0007,800137-0007,Yes
1,2,HIMACHAL PRADESH,33,Shimla,182,Shimla (urban),800137,17,Shimla (M Corp.) WARD NO.-0017,Urban,6526,169578,6864602.0,033-00182-800137-0017,800137-0017,Yes
2,2,HIMACHAL PRADESH,33,Shimla,182,Shimla (urban),800137,25,Shimla (M Corp.) WARD NO.-0025,Urban,6036,169578,6864602.0,033-00182-800137-0025,800137-0025,Yes
3,3,PUNJAB,37,Jalandhar,212,Jalandhar - I,800166,5,Jalandhar (M Corp.) WARD NO.-0005,Urban,11500,1145692,27743338.0,037-00212-800166-0005,800166-0005,Yes
4,3,PUNJAB,37,Jalandhar,212,Jalandhar - I,800166,10,Jalandhar (M Corp.) WARD NO.-0010,Urban,21042,1145692,27743338.0,037-00212-800166-0010,800166-0010,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
683,90,TELANGANA,537,Rangareddy,4524,Uppal,802918,5,GHMC (M Corp.) (Part) WARD NO.-0005,Urban,57688,384835,NaN,537-04524-802918-0005,802918-0005,Yes
684,90,TELANGANA,537,Rangareddy,4524,Uppal,802918,8,GHMC (M Corp.) (Part) WARD NO.-0008,Urban,75227,384835,NaN,537-04524-802918-0008,802918-0008,Yes
685,90,TELANGANA,538,Mahbubnagar,4568,Mahbubnagar,802922,6,Mahbubnagar (M) WARD NO.-0006,Urban,10515,249091,NaN,538-04568-802922-0006,802922-0006,Yes
686,90,TELANGANA,541,Khammam,4757,Khammam (Urban),579675,0,Erlapudi,Rural,12023,313504,NaN,541-04757-579675-0000,579675-0000,Yes


In [11]:
panel_df.to_csv(CLEANED_DATA_DIR / "Panel Wards.csv", index=False)

In [12]:
# get unique across district and subdistrict both
len(panel_df[["State", "District", "Subdistrict"]].drop_duplicates())

242

#### Check for duplicated wards in our own sample

In [13]:
duplicated_sampled_wards = panel_df[
    panel_df[["TownVillage", "UrbanWardVillage"]].duplicated(keep=False)
].sort_values(["TownVillage", "UrbanWardVillage"])
duplicated_sampled_wards

,State,State_Name,District,District_Name,Subdistrict,Subd_Name,TownVillage,UrbanWardVillage,WardVillage_Name,TRU,WardVillage_Pop,Subd_Pop,State_Pop,WardVillageID,PCA_ID,Ward Boundary Available with MapSolve


In [14]:
duplicated_sampled_wards.to_csv(
    CLEANED_DATA_DIR / "Duplicated Sampled Wards.csv", index=False
)

In [15]:
duplicated_sampled_wards

,State,State_Name,District,District_Name,Subdistrict,Subd_Name,TownVillage,UrbanWardVillage,WardVillage_Name,TRU,WardVillage_Pop,Subd_Pop,State_Pop,WardVillageID,PCA_ID,Ward Boundary Available with MapSolve


## 1. Load all boundaries

In [16]:
# get all filepaths that end in `gpkg` inside the RAW_DATA_DIR / 0.1. MapSolve Boundaries
boundaries_dir = RAW_DATA_DIR / "01. MapSolve Boundaries"
gpkg_files_all = list(boundaries_dir.glob("**/*.gpkg"))
gpkg_files_all = [f for f in gpkg_files_all if f.is_file()]

In [17]:
# load all shapes into one gdf
gdf_list = []
for filepath in gpkg_files_all:
    gdf_list.append(gpd.read_file(filepath))
gdf = pd.concat(gdf_list, ignore_index=True)

In [18]:
gdf = gdf.drop_duplicates()

#### Fix Chittoor issue

In [19]:
# gdf[gdf["Dist_N"] == "Chittoor"]

In [20]:
gdf.loc[gdf["TV_C"] == 803014, ["SubDist_N", "SubDist_C"]] = ["Tirupati (Urban)", 5383]

In [21]:
# gdf[gdf["Dist_N"] == "Chittoor"]

#### Fix Patna ward PCA_ID issue

In [22]:
# gdf.loc[(gdf["TV_C"] == 801373) & (gdf["Ward_C"] == "3")]

In [23]:
gdf.loc[(gdf["TV_C"] == 801373) & (gdf["Ward_C"] == "3"), "PCA_ID"] = "801373-3"

In [24]:
# gdf.loc[(gdf["TV_C"] == 801373) & (gdf["Ward_C"] == "3")]

#### Replace some MDDS codes with TV and Ward codes as census expects

In [25]:
# gdf.loc[(gdf["TV_C"].isin([519923, 574077]))]

In [26]:
gdf.loc[gdf["TV_C"] == 519923, ["TV_C", "Ward_C", "PCA_ID"]] = [802596, 18, "802596-18"]
gdf.loc[gdf["TV_C"] == 574077, ["TV_C", "Ward_C", "PCA_ID"]] = [
    802918,
    157,
    "802918-157",
]

## 2. Checks

### Any MapSolve rows with missing TV code?

In [27]:
gdf_no_TV_code_filtered = gdf[gdf["TV_C"].isna()]
gdf_no_TV_code_filtered

,UID,PCA_ID,State_C,State_N,Dist_C,Dist_N,SubDist_C,SubDist_N,TV_C,TV_N,Ward_C,TOT_P,geometry
24,16,None,22.0,Chhattisgarh,409.0,Durg,3317.0,Durg,NaN,None,NaN,NaN,"MULTIPOLYGON (((9037170.000 2420040.000, 90371..."
25,16,None,22.0,Chhattisgarh,406.0,Bilaspur,3295.0,Bilaspur,NaN,None,NaN,NaN,"MULTIPOLYGON (((9166590.000 2550750.000, 91665..."
1460,77,None,23.0,Madhya Pradesh,451.0,Jabalpur,3631.0,Jabalpur,NaN,None,NaN,NaN,"POLYGON ((8900430.000 2632320.000, 8900490.000..."
1658,9504321,55658,5.0,Uttarakhand,67.0,Udham Singh Nagar,346.0,Kashipur,NaN,None,NaN,283136.0,"MULTIPOLYGON (((8796240.000 3417120.000, 87963..."
1659,9488658,45145,5.0,Uttarakhand,60.0,Dehradun,304.0,Dehradun,NaN,None,NaN,988007.0,"MULTIPOLYGON (((8678280.000 3566610.000, 86783..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
47977,240,7-95-447,7.0,NCT of Delhi,95.0,Central,447.0,Karol Bagh,NaN,NaN,NaN,136599.0,"MULTIPOLYGON (((8591760.000 3331200.000, 85918..."
47978,219,7-98-454,7.0,NCT of Delhi,98.0,South,454.0,Hauz Khas (Saket),NaN,NaN,NaN,1219100.0,"MULTIPOLYGON (((8595720.000 3300750.000, 85957..."
47979,155,7-97-453,7.0,NCT of Delhi,97.0,South West,453.0,Vasant Vihar,NaN,NaN,NaN,641666.0,"MULTIPOLYGON (((8586120.000 3312960.000, 85861..."
47980,156,7-97-452,7.0,NCT of Delhi,97.0,South West,452.0,Delhi Cantonment,NaN,NaN,NaN,286140.0,"MULTIPOLYGON (((8589120.000 3321540.000, 85891..."


In [28]:
gdf_no_TV_code_filtered.to_csv(
    CLEANED_DATA_DIR / "MapSolve Data without TV Codes.csv", index=False
)

### Request satisfaction check

In [29]:
given_states_list = list(gdf["State_C"].unique())
given_states_list.append(
    90
)  # manually add 90 for Telangana / Andhra Pradesh discrepency
len(given_states_list)

24

In [30]:
panel_df.loc[panel_df["State"].isin(given_states_list), "State_Name"].unique()

array(['HIMACHAL PRADESH', 'PUNJAB', 'UTTARAKHAND', 'HARYANA',
       'NCT OF DELHI', 'RAJASTHAN', 'UTTAR PRADESH', 'BIHAR', 'TRIPURA',
       'ASSAM', 'WEST BENGAL', 'JHARKHAND', 'ODISHA', 'CHHATTISGARH',
       'MADHYA PRADESH', 'GUJARAT', 'MAHARASHTRA', 'ANDHRA PRADESH',
       'KARNATAKA', 'KERALA', 'TAMIL NADU', 'TELANGANA'], dtype=object)

In [31]:
panel_df["State Shared by MapSolve"] = False
panel_df.loc[panel_df["State"].isin(given_states_list), "State Shared by MapSolve"] = (
    True
)

In [32]:
len(panel_df[~panel_df["State Shared by MapSolve"]])

0

In [33]:
panel_df["State Changed"] = "No"
panel_df.loc[panel_df["State_Name"] == "TELANGANA", "State Changed"] = (
    "Previously Andhra Pradesh"
)

In [34]:
panel_df

,State,State_Name,District,District_Name,Subdistrict,Subd_Name,TownVillage,UrbanWardVillage,WardVillage_Name,TRU,WardVillage_Pop,Subd_Pop,State_Pop,WardVillageID,PCA_ID,Ward Boundary Available with MapSolve,State Shared by MapSolve,State Changed
0,2,HIMACHAL PRADESH,33,Shimla,182,Shimla (urban),800137,7,Shimla (M Corp.) WARD NO.-0007,Urban,8205,169578,6864602.0,033-00182-800137-0007,800137-0007,Yes,True,No
1,2,HIMACHAL PRADESH,33,Shimla,182,Shimla (urban),800137,17,Shimla (M Corp.) WARD NO.-0017,Urban,6526,169578,6864602.0,033-00182-800137-0017,800137-0017,Yes,True,No
2,2,HIMACHAL PRADESH,33,Shimla,182,Shimla (urban),800137,25,Shimla (M Corp.) WARD NO.-0025,Urban,6036,169578,6864602.0,033-00182-800137-0025,800137-0025,Yes,True,No
3,3,PUNJAB,37,Jalandhar,212,Jalandhar - I,800166,5,Jalandhar (M Corp.) WARD NO.-0005,Urban,11500,1145692,27743338.0,037-00212-800166-0005,800166-0005,Yes,True,No
4,3,PUNJAB,37,Jalandhar,212,Jalandhar - I,800166,10,Jalandhar (M Corp.) WARD NO.-0010,Urban,21042,1145692,27743338.0,037-00212-800166-0010,800166-0010,Yes,True,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
683,90,TELANGANA,537,Rangareddy,4524,Uppal,802918,5,GHMC (M Corp.) (Part) WARD NO.-0005,Urban,57688,384835,NaN,537-04524-802918-0005,802918-0005,Yes,True,Previously Andhra Pradesh
684,90,TELANGANA,537,Rangareddy,4524,Uppal,802918,8,GHMC (M Corp.) (Part) WARD NO.-0008,Urban,75227,384835,NaN,537-04524-802918-0008,802918-0008,Yes,True,Previously Andhra Pradesh
685,90,TELANGANA,538,Mahbubnagar,4568,Mahbubnagar,802922,6,Mahbubnagar (M) WARD NO.-0006,Urban,10515,249091,NaN,538-04568-802922-0006,802922-0006,Yes,True,Previously Andhra Pradesh
686,90,TELANGANA,541,Khammam,4757,Khammam (Urban),579675,0,Erlapudi,Rural,12023,313504,NaN,541-04757-579675-0000,579675-0000,Yes,True,Previously Andhra Pradesh


#### Check for wards

In [35]:
panel_df["PCA_ID"] = (
    panel_df["TownVillage"].astype(str) + "-" + panel_df["UrbanWardVillage"].astype(str)
)
given_ward_ids = gdf["PCA_ID"].unique()

panel_df["Ward Boundary Given"] = False
panel_df.loc[panel_df["PCA_ID"].isin(given_ward_ids), "Ward Boundary Given"] = True

In [36]:
len(set(panel_df["PCA_ID"]).intersection(given_ward_ids))

298

#### Check for TownVillage codes

In [37]:
given_TV_ids = set(
    gdf.loc[
        gdf["Ward_C"].isna() & gdf["TV_C"].notna(),
        "TV_C",
    ].astype(int)
)

panel_df["TV Boundary Given"] = False
panel_df.loc[
    panel_df["TownVillage"].astype(int).isin(given_TV_ids),
    "TV Boundary Given",
] = True

#### Check for SubDistricts

In [38]:
given_subdist_ids = set(
    gdf.loc[
        gdf["Ward_C"].isna() & gdf["TV_C"].isna() & gdf["SubDist_C"].notna(),
        "SubDist_C",
    ].astype(int)
)

panel_df["SubDistrict Boundary Given"] = False
panel_df.loc[
    panel_df["Subdistrict"].astype(int).isin(given_subdist_ids),
    "SubDistrict Boundary Given",
] = True

#### Fill in Delivery State

- BETTER - Ward boundary given but only TV or Subdistrict was expected
- GOOD - Ward boundary given as expected
- GOOD - Town/Village boundary given as expected
- GOOD - Subdistrict boundary given as expected
- BAD - Town/Village boundary given but Ward boundaries expected
- BAD - Subdistrict boundary given but Ward boundaries expected
- BAD - No boundary(s) given

In [39]:
## baseline
panel_df["Delivery State"] = "BAD - No boundary(s) given"

## subdistrict
panel_df.loc[
    (panel_df["Ward Boundary Available with MapSolve"] == "No")
    & panel_df["SubDistrict Boundary Given"],
    "Delivery State",
] = "GOOD - Subdistrict boundary given as expected"
panel_df.loc[
    (panel_df["Ward Boundary Available with MapSolve"] == "Yes")
    & panel_df["SubDistrict Boundary Given"],
    "Delivery State",
] = "BAD - Subdistrict boundary given but Ward boundaries expected"

# tripura > dukli special case
panel_df.loc[
    (panel_df["State_Name"] == "TRIPURA") & (panel_df["Subd_Name"] == "Dukli"),
    "Delivery State",
] = "GOOD - Subdistrict boundary given as expected"

## town/village
panel_df.loc[
    (panel_df["Ward Boundary Available with MapSolve"] == "No")
    & panel_df["TV Boundary Given"],
    "Delivery State",
] = "GOOD - Town/Village boundary given as expected"
panel_df.loc[
    (panel_df["Ward Boundary Available with MapSolve"] == "Yes")
    & panel_df["TV Boundary Given"],
    "Delivery State",
] = "BAD - Town/Village boundary given but Ward boundaries expected"

## ward
panel_df.loc[
    (panel_df["Ward Boundary Available with MapSolve"] == "No")
    & panel_df["Ward Boundary Given"],
    "Delivery State",
] = "BETTER - Ward boundary given but only TV or Subdistrict was expected"
panel_df.loc[
    (panel_df["Ward Boundary Available with MapSolve"] == "Yes")
    & panel_df["Ward Boundary Given"],
    "Delivery State",
] = "GOOD - Ward boundary given as expected"

In [40]:
panel_df["Delivery State"].value_counts()

Delivery State
GOOD - Ward boundary given as expected                            298
BAD - Town/Village boundary given but Ward boundaries expected    276
BAD - Subdistrict boundary given but Ward boundaries expected      64
BAD - No boundary(s) given                                         48
GOOD - Subdistrict boundary given as expected                       2
Name: count, dtype: int64

#### Add `PSU Type` column

In [41]:
psu_mapping = {
    "BETTER - Ward boundary given but only TV or Subdistrict was expected": "ward",
    "GOOD - Town/Village boundary given as expected": "town_village",
    "GOOD - Ward boundary given as expected": "ward",
    "GOOD - Subdistrict boundary given as expected": "subdistrict",
    "BAD - Town/Village boundary given but Ward boundaries expected": "town_village",
    "BAD - No boundary(s) given": "none",
    "BAD - Subdistrict boundary given but Ward boundaries expected": "subdistrict",
}
# create a new column in panel_df called PSU Type
panel_df["PSU Type"] = panel_df["Delivery State"].map(psu_mapping)

In [42]:
panel_df["PSU Type"].fillna("none", inplace=True)

In [43]:
panel_df["PSU Type"].value_counts()

PSU Type
ward            298
town_village    276
subdistrict      66
none             48
Name: count, dtype: int64

#### Add `PSU ID` column (unique overall ID)

In [44]:
# add an ID column that is unique across all rows. It should be WARD_{PCA_ID} if the PSU Type is "ward", TV_{TV Code} if the PSU Type is "town_village", and SUBDISTRICT_{Subdistrict Code} if the PSU Type is "subdistrict"
panel_df["PSU ID"] = panel_df.apply(
    lambda row: f"WARD_{row['PCA_ID']}"
    if row["PSU Type"] == "ward"
    else f"TV_{int(row['TownVillage'])}"
    if row["PSU Type"] == "town_village"
    else f"SUBDISTRICT_{int(row['Subdistrict'])}",
    axis=1,
).astype(str)

#### Add `Ward Count` column

In [45]:
panel_df["Ward Count"] = 0

panel_df.loc[panel_df["PSU Type"] == "ward", "Ward Count"] = 1
panel_df.loc[panel_df["PSU Type"] == "town_village", "Ward Count"] = (
    panel_df[panel_df["PSU Type"] == "town_village"]
    .groupby("TownVillage")["UrbanWardVillage"]
    .transform("count")
)
panel_df.loc[panel_df["PSU Type"] == "subdistrict", "Ward Count"] = (
    panel_df[panel_df["PSU Type"] == "subdistrict"]
    .groupby("Subdistrict")["UrbanWardVillage"]
    .transform("count")
)
panel_df["Ward Count"] = panel_df["Ward Count"].astype(int)

#### Reorder columns

In [46]:
reordered_columns = [
    "State",
    "State_Name",
    "District",
    "District_Name",
    "Subdistrict",
    "Subd_Name",
    "TownVillage",
    "UrbanWardVillage",
    "WardVillage_Name",
    "PCA_ID",
    "TRU",
    "WardVillage_Pop",
    "Subd_Pop",
    "State_Pop",
    "WardVillageID",
    "Ward Boundary Available with MapSolve",
    "State Shared by MapSolve",
    "State Changed",
    "Ward Boundary Given",
    "TV Boundary Given",
    "SubDistrict Boundary Given",
    "Delivery State",
    "PSU Type",
    "PSU ID",
    "Ward Count",
]
panel_df = panel_df[reordered_columns]

#### Save

In [47]:
panel_df.to_csv(CLEANED_DATA_DIR / "Panel Wards with Quality Checks.csv", index=False)

In [48]:
if not panel_df["State Shared by MapSolve"].all():
    print("There are states in the merged data that are not shared by MapSolve:")
    print(panel_df[~panel_df["State Shared by MapSolve"]]["State"].unique())
    print("Saving the states for which we DO have data separately:")
    panel_df[panel_df["State Shared by MapSolve"]].to_csv(
        CLEANED_DATA_DIR / "Panel Wards with Quality Checks - Shared States.csv",
        index=False,
    )

#### Save per-state stats

In [49]:
# Get counts of delivery states by state
delivery_state_counts = (
    panel_df[panel_df["State"].isin(given_states_list)]
    .groupby("State")["Delivery State"]
    .value_counts()
)

# Convert to a more readable DataFrame format
delivery_state_pivot = delivery_state_counts.unstack(fill_value=0).reset_index()

# Sort by state code
delivery_state_pivot = delivery_state_pivot.sort_values(by="State")

# Add state names for better readability
state_name_mapping = (
    panel_df[["State", "State_Name"]].drop_duplicates().set_index("State")["State_Name"]
)
delivery_state_pivot["State_Name"] = delivery_state_pivot["State"].map(
    state_name_mapping
)

# Reorder columns to show State_Name first
columns = ["State", "State_Name"] + [
    col for col in delivery_state_pivot.columns if col not in ["State", "State_Name"]
]
delivery_state_pivot = delivery_state_pivot[columns]

# transform the DataFrame to have the delivery states as rows and state names as columns
delivery_state_pivot.drop(columns=["State"], inplace=True)
delivery_state_pivot = delivery_state_pivot.set_index("State_Name").T.reset_index()
delivery_state_pivot

State_Name,Delivery State,HIMACHAL PRADESH,PUNJAB,UTTARAKHAND,HARYANA,NCT OF DELHI,RAJASTHAN,UTTAR PRADESH,BIHAR,TRIPURA,...,ODISHA,CHHATTISGARH,MADHYA PRADESH,GUJARAT,MAHARASHTRA,ANDHRA PRADESH,KARNATAKA,KERALA,TAMIL NADU,TELANGANA
0,BAD - No boundary(s) given,0,0,0,1,0,0,10,11,0,...,2,0,0,0,3,5,0,0,1,0
1,BAD - Subdistrict boundary given but Ward boun...,0,0,32,0,30,0,0,0,0,...,0,2,0,0,0,0,0,0,0,0
2,BAD - Town/Village boundary given but Ward bou...,0,9,0,13,2,2,28,20,22,...,7,12,7,7,11,19,2,23,19,10
3,GOOD - Subdistrict boundary given as expected,0,0,0,0,0,0,0,0,2,...,0,0,0,0,0,0,0,0,0,0
4,GOOD - Ward boundary given as expected,3,20,1,8,46,7,10,8,0,...,9,5,10,34,25,0,31,9,21,21


In [50]:
# add a total column as the second column
delivery_state_pivot.insert(1, "Total", delivery_state_pivot.iloc[:, 1:].sum(axis=1))

In [51]:
delivery_state_pivot

State_Name,Delivery State,Total,HIMACHAL PRADESH,PUNJAB,UTTARAKHAND,HARYANA,NCT OF DELHI,RAJASTHAN,UTTAR PRADESH,BIHAR,...,ODISHA,CHHATTISGARH,MADHYA PRADESH,GUJARAT,MAHARASHTRA,ANDHRA PRADESH,KARNATAKA,KERALA,TAMIL NADU,TELANGANA
0,BAD - No boundary(s) given,48,0,0,0,1,0,0,10,11,...,2,0,0,0,3,5,0,0,1,0
1,BAD - Subdistrict boundary given but Ward boun...,64,0,0,32,0,30,0,0,0,...,0,2,0,0,0,0,0,0,0,0
2,BAD - Town/Village boundary given but Ward bou...,276,0,9,0,13,2,2,28,20,...,7,12,7,7,11,19,2,23,19,10
3,GOOD - Subdistrict boundary given as expected,2,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,GOOD - Ward boundary given as expected,298,3,20,1,8,46,7,10,8,...,9,5,10,34,25,0,31,9,21,21


In [52]:
delivery_state_pivot.to_csv(
    CLEANED_DATA_DIR / "Delivery State Counts By State.csv", index=False
)